## Modelo final

En la etapa anterior se compararon los modelos de Regresión Logística y Random Forest, seleccionándose este último por presentar el mejor desempeño sobre el conjunto de prueba. En esta notebook se realiza el reentrenamiento del modelo utilizando la totalidad de los datos de `temporada1.parquet`. Posteriormente, el modelo ajustado se emplea para generar las predicciones correspondientes a `temporada2.parquet`.

In [1]:
# Cargamos los paquetes necesarios
import matplotlib.pyplot as plt 
import polars as pl
import pyprojroot
from plotnine import *
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score, 
    roc_curve,
    log_loss,
    f1_score
)

In [ ]:
# Definir la ruta raiz del proyecto
ROOT = pyprojroot.here()

# Cargamos los datos
datos_entrenamiento = pl.read_parquet(ROOT / "datos" / "temporada1.parquet")
datos_validacion = pl.read_parquet(ROOT / "datos" / "temporada2.parquet")

# Convertimos a DataFrame de pandas
datos_entrenamiento_pd = datos_entrenamiento.to_pandas()
datos_validacion_pd = datos_validacion.to_pandas()

In [ ]:
# Variables seleccionadas por LASSO
variables = [
    "altura_zona",
    "plate_x",
    "plate_z",
    "pitch_type",
    "release_speed",
    "balls",
    "strikes"
]

In [ ]:
# Se define la matriz X y el vector y para los datos de entrenamiento
X_train = datos_entrenamiento_pd[variables]
y_train = datos_entrenamiento_pd["swing"]

# Se define la matriz X y el vector y para los datos de validación
X_test = datos_validacion_pd[variables]
y_test = datos_validacion_pd["swing"]

In [ ]:
# Se convierten las variables categóricas a dummies para ambos conjuntos de datos
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

In [ ]:
# Nos aseguramos que los dos data sets tengan las mismas variables
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

### Reentrenamiento del modelo

Se realiza un nuevo ajuste utilizando la totalidad de los datos disponibles en `temporada1.parquet`. En esta etapa ya no se divide el conjunto en entrenamiento y prueba, sino que se aprovecha toda la información para obtener un modelo definitivo antes de generar las predicciones sobre `temporada2.parquet`.


In [ ]:
# Se crea el modelo con Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=123
)

In [ ]:
# Se ajusta el modelo
rf.fit(X_train, y_train)

### Predicción sobre el conjunto de validación

Una vez reentrenado el modelo con la totalidad de los datos contenidos en `temporada1.parquet`, se generan las predicciones sobre `temporada2.parquet`, correspondiente al conjunto de validación principal del trabajo. El objetivo de esta etapa es predecir, para cada lanzamiento, si el bateador realizará o no un *swing*.